# Ref-Injection Dataset Generation

Converts a QA dataset (FinanceQA-style, or any HF dataset with a question/answer
column) into training episodes for the PersonaPlex `<ref>...</ref>` context-injection
LoRA.

**Workflow this notebook implements:**

```
QA dataset (HF hub)
   -> for each row: LLM rewrites it into
        - a spoken-style question
        - a compressed one-sentence "ref_fact" (matching the production
          compressor's own style rules)
        - a natural spoken answer that actually uses that fact
   -> paired with synthetic "scope negative" and "no context" episodes
   -> validated (tag-leak check, grounding-overlap check, contamination check)
   -> written out as JSONL, split into train/val
```

Run this **first**, then feed its output (`dataset_out/train.jsonl`,
`dataset_out/val.jsonl`) into `02_LoRA_Training.ipynb`.

This notebook runs a local instruct model (default Qwen/Qwen2.5-14B-Instruct)
on this pod's own GPU to do the conversion -- no external API, no API key. It
never loads PersonaPlex itself, so it can run on a separate, smaller pod from
the training notebook if you prefer.

## 1. Environment setup

In [ ]:
# Qwen2.5 needs a reasonably modern transformers -- an older/mismatched
# transformers + tokenizers + accelerate combination is what produces
# "ModuleNotFoundError: Could not import module 'Qwen2ForCausalLM'" at load
# time. These versions match transformers==4.52.4, the same pin
# prepare_imtalker_personaplex.sh already uses for the (also Qwen2.5-family)
# compressor model in the live pipeline, so they're known-good on this repo.
%pip install -q "transformers==4.52.4" "tokenizers>=0.21,<0.22" "accelerate>=0.34" \
    datasets bitsandbytes safetensors sentencepiece huggingface_hub --upgrade
print("Environment ready.")

In [ ]:
import torch, transformers
print("torch:", torch.__version__, " cuda available:", torch.cuda.is_available())
print("transformers:", transformers.__version__)
assert tuple(int(x) for x in transformers.__version__.split(".")[:2]) >= (4, 46), (
    f"transformers {transformers.__version__} is too old for Qwen2.5 -- re-run the "
    f"pip install cell above (or restart the kernel first if you just upgraded it)."
)

In [ ]:
import sys, os
from pathlib import Path

# This notebook lives in <project_root>/ref_lora_training/. Add the project
# root to sys.path so `ref_lora_training.common...` imports work regardless
# of the notebook's working directory, and so `common.ref_format` can find
# `IMTalker/search_helpers.py` for the production tag-wrapping helpers.
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "IMTalker").exists():
    # e.g. running from inside ref_lora_training/ itself
    PROJECT_ROOT = PROJECT_ROOT.parent
assert (PROJECT_ROOT / "IMTalker").exists(), (
    f"Could not find the IMTalker/ folder from {PROJECT_ROOT}. "
    "Run this notebook from inside the project (or ref_lora_training/), "
    "or edit PROJECT_ROOT above by hand."
)
sys.path.insert(0, str(PROJECT_ROOT))
print("PROJECT_ROOT =", PROJECT_ROOT)

## 2. Configuration

Edit this cell for your run. Everything below reads from it.

In [ ]:
# --- QA source dataset -------------------------------------------------
# Default is a real, popular finance QA dataset (question/answer/context
# columns, derived from 10-K filings). Swap DATASET_HF_ID + the column names
# for any other QA dataset (TriviaQA, SQuAD, NaturalQuestions, your own JSON
# loaded via datasets.load_dataset("json", data_files=...), etc.) -- nothing
# else in this notebook needs to change if you keep the same three columns.
DATASET_HF_ID = "virattt/financial-qa-10K"
DATASET_SPLIT = "train"
QUESTION_COL = "question"
ANSWER_COL = "answer"
CONTEXT_COL = "context"          # set to "" if your dataset has no context column
N_SOURCE_ROWS = 600              # how many QA rows to convert into grounded episodes
RANDOM_SEED = 42

# --- LLM used to convert QA rows into our training format ---------------
# Local only: runs on this pod's GPU, no API key, nothing leaves the pod.
LOCAL_MODEL_ID = "Qwen/Qwen2.5-14B-Instruct"
LOCAL_4BIT = True
GENERATOR_TEMPERATURE = 0.4
GENERATOR_MAX_TOKENS = 600       # headroom for a verbose ref_fact + spoken_answer without truncating
# N_GPUS = None -> auto-detect every visible GPU and run one worker process per
# GPU, each with its own full copy of the model, splitting the QA rows between
# them (data parallelism -- the right kind of parallelism for many independent,
# short generations, and what actually drives every GPU to ~100% utilization).
# Set to an int to cap how many GPUs are used.
N_GPUS = None

# --- Output ---------------------------------------------------------------
OUTPUT_DIR = PROJECT_ROOT / "ref_lora_training" / "dataset_out"
VAL_RATIO = 0.08

# --- Dataset composition ---------------------------------------------------
# For every N grounded episodes successfully generated, also synthesize:
SCOPE_NEGATIVE_RATIO = 0.35   # fraction of grounded episodes that get a paired
                              # "unrelated follow-up must not reuse the fact" episode
N_NO_CONTEXT_EPISODES = 40    # template-based "search found nothing" examples
N_NO_REF_BASELINE_EPISODES = 60  # ordinary no-<ref> chat, keeps general ability intact

print("Config loaded.")

## 3. Load QA rows from the source dataset

Prints a couple of raw rows first -- **check that `QUESTION_COL`/`ANSWER_COL`/
`CONTEXT_COL` actually look right for your dataset** before spending LLM calls
on the full run.

In [ ]:
from datasets import load_dataset

_preview = load_dataset(DATASET_HF_ID, split=f"{DATASET_SPLIT}[:3]")
for i, row in enumerate(_preview):
    print(f"--- row {i} ---")
    print("question:", str(row.get(QUESTION_COL))[:200])
    print("answer  :", str(row.get(ANSWER_COL))[:200])
    if CONTEXT_COL:
        print("context :", str(row.get(CONTEXT_COL))[:200])
    print()

In [ ]:
from ref_lora_training.common.dataset_builder import load_qa_rows

qa_rows = load_qa_rows(
    dataset_id=DATASET_HF_ID, split=DATASET_SPLIT,
    question_col=QUESTION_COL, answer_col=ANSWER_COL, context_col=CONTEXT_COL,
    limit=N_SOURCE_ROWS, seed=RANDOM_SEED,
)
print(f"loaded {len(qa_rows)} usable QA rows")

## 4. Convert QA rows into grounded episodes via the LLM

Each row becomes one `ExampleType.GROUNDED` episode: a spoken-style question,
a compressed `<ref>`-ready fact (written with the same rules the production
`ContextCompressor` uses), and a natural spoken reply that actually uses it.

This is the slow, GPU-bound step -- it runs across every visible GPU (see below).

Runs one worker process **per visible GPU**, each loading its own copy of the model and working through its own slice of `qa_rows` in parallel -- with N GPUs this is roughly an N x speedup over a single GPU, and every GPU should show ~100% utilization in `nvidia-smi` while this runs. Output from every worker streams live below, prefixed `[gpu 0]`, `[gpu 1]`, etc. Safe to interrupt and re-run -- each GPU's shard output file resumes on its own.

In [ ]:
from ref_lora_training.common.multi_gpu_runner import run_sharded_generation, detect_gpu_count
from ref_lora_training.common.dataset_builder import write_jsonl, read_jsonl
from ref_lora_training.common.ref_format import DEFAULT_SYSTEM_PROMPT

n_gpus = N_GPUS or detect_gpu_count()
print(f"detected/using {n_gpus} GPU(s)")

raw_path = OUTPUT_DIR / "raw_generated.jsonl"
source_tag = DATASET_HF_ID.split("/")[-1]

grounded_episodes = run_sharded_generation(
    qa_rows=qa_rows,
    local_model_id=LOCAL_MODEL_ID,
    local_4bit=LOCAL_4BIT,
    temperature=GENERATOR_TEMPERATURE,
    max_tokens=GENERATOR_MAX_TOKENS,
    system_prompt=DEFAULT_SYSTEM_PROMPT,
    source_tag=source_tag,
    output_dir=OUTPUT_DIR,
    n_gpus=n_gpus,
    hf_token=os.getenv("HF_TOKEN", ""),
)
write_jsonl(grounded_episodes, raw_path)
print(f"done: {len(grounded_episodes)} grounded episodes written to {raw_path}")

## 5. Validate the grounded episodes

Runs the same checks `common/ref_format.validate_episode` performs: does the
reply actually mention the injected fact (not just ignore it), and does it
leak a literal `<ref>`/`<lookup>` tag into the spoken text. Flagged rows are
printed for a spot check and then dropped -- a bad training example doesn't
just fail to help, it actively teaches the wrong behavior.

In [ ]:
from ref_lora_training.common.dataset_builder import validate_all

report = validate_all(grounded_episodes)
print(f"clean: {report['clean']} / {report['total']}")
for flagged in report["flagged"][:15]:
    print(" -", flagged["id"], flagged["problems"])
if len(report["flagged"]) > 15:
    print(f"   ... and {len(report['flagged']) - 15} more")

flagged_ids = {f["id"] for f in report["flagged"]}
clean_grounded = [ep for ep in grounded_episodes if ep.id not in flagged_ids]
print(f"keeping {len(clean_grounded)} clean grounded episodes")

## 6. Synthesize scope-negative, no-context, and no-ref-baseline episodes

No LLM calls here -- these are built directly from `common/distractors.py` and
fixed templates. This is what specifically targets the cross-turn
contamination bug (a `<ref>` fact bleeding into unrelated later turns) that
was observed in production: every `scope_negative` episode pairs a real
grounded exchange with a second, unrelated turn that must NOT reuse the fact.

In [ ]:
import random
from ref_lora_training.common.dataset_builder import (
    build_scope_negative_episode, build_no_context_episode, build_no_ref_baseline_episode,
)

rng = random.Random(RANDOM_SEED)
n_scope_neg = int(len(clean_grounded) * SCOPE_NEGATIVE_RATIO)
scope_negative_episodes = [
    build_scope_negative_episode(ep, i, rng)
    for i, ep in enumerate(rng.sample(clean_grounded, n_scope_neg))
]
no_context_episodes = [
    build_no_context_episode(i, DEFAULT_SYSTEM_PROMPT, rng) for i in range(N_NO_CONTEXT_EPISODES)
]
no_ref_baseline_episodes = [
    build_no_ref_baseline_episode(i, DEFAULT_SYSTEM_PROMPT, rng) for i in range(N_NO_REF_BASELINE_EPISODES)
]
print(f"scope_negative: {len(scope_negative_episodes)}")
print(f"no_context:     {len(no_context_episodes)}")
print(f"no_ref_baseline:{len(no_ref_baseline_episodes)}")

## 7. Eyeball a few finished episodes before committing to them

In [ ]:
from ref_lora_training.common.ref_format import render_episode_text

all_episodes = clean_grounded + scope_negative_episodes + no_context_episodes + no_ref_baseline_episodes
random.Random(0).shuffle(all_episodes)
for ep in all_episodes[:6]:
    print(f"===== {ep.id}  [{ep.example_type}] =====")
    print(render_episode_text(ep, include_system_prompt=False))
    print()

## 8. Write final train/val split

In [ ]:
from ref_lora_training.common.dataset_builder import train_val_split

train_eps, val_eps = train_val_split(all_episodes, val_ratio=VAL_RATIO, seed=RANDOM_SEED)
n_train = write_jsonl(train_eps, OUTPUT_DIR / "train.jsonl")
n_val = write_jsonl(val_eps, OUTPUT_DIR / "val.jsonl")

from collections import Counter
print(f"train: {n_train} episodes  {dict(Counter(e.example_type for e in train_eps))}")
print(f"val:   {n_val} episodes  {dict(Counter(e.example_type for e in val_eps))}")
print()
print(f"Wrote:\n  {OUTPUT_DIR / 'train.jsonl'}\n  {OUTPUT_DIR / 'val.jsonl'}")
print("\nNext: open 02_LoRA_Training.ipynb and point DATASET_DIR at this ref_lora_training/dataset_out/ folder.")